# Day D2 — 전처리와 QC: slice timing, 움직임, 왜곡, smoothing

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/D-fmri/day-D2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/D-fmri/day-D2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — interleaved 획득 시각과 aliasing 계산

In [ ]:
import numpy as np

# (가) interleaved 슬라이스 획득 시각: TR 안에서 슬라이스마다 찍히는 시각이 다르다
TR, n_slices = 1.5, 10
order = list(range(0, n_slices, 2)) + list(range(1, n_slices, 2))   # 0,2,4,...,1,3,5,...
t_slice = np.empty(n_slices)
t_slice[order] = np.arange(n_slices) * TR / n_slices
print("획득 순서:", order)
print("슬라이스별 획득 시각(s):", np.round(t_slice, 2))
print("이웃한 슬라이스 0과 1의 시간 차:", t_slice[1] - t_slice[0], "s")

# (나) aliasing: TR보다 빠른 심장 박동은 느린 가짜 리듬으로 보인다
def aliased(f_true, fs):
    return abs(f_true - fs * round(f_true / fs))

for TR in [0.5, 1.5, 2.0]:
    fs = 1 / TR
    print(f"TR={TR}s (Nyquist {fs/2:.2f} Hz): 심박 1.2 Hz → {aliased(1.2, fs):.2f} Hz,"
          f" 호흡 0.3 Hz → {aliased(0.3, fs):.2f} Hz")

### 예시 2 — 움직임 추정치에서 framewise displacement 계산

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(7)
n = 150
# 가상 움직임 추정치: 3개 이동(mm) + 3개 회전(rad). 천천히 흐르다가 두 번 '움찔'한다
trans = np.cumsum(rng.normal(0, 0.03, size=(n, 3)), axis=0)
rot = np.cumsum(rng.normal(0, 0.0004, size=(n, 3)), axis=0)
trans[60:, 2] += 0.9          # 60번째 시점에서 z 방향으로 0.9 mm 갑자기 이동
rot[100:, 0] += 0.012         # 100번째 시점에서 x축으로 약 0.7도 회전
motion = pd.DataFrame(np.hstack([trans, rot]),
                      columns=["trans_x", "trans_y", "trans_z", "rot_x", "rot_y", "rot_z"])

# framewise displacement(FD): 이웃한 시점 사이 변화량의 절댓값 합.
# 회전은 반지름 50 mm 구 표면의 이동 거리로 바꿔 더한다 (Power 등 2012의 정의)
d = motion.diff().abs().fillna(0)
fd = d[["trans_x", "trans_y", "trans_z"]].sum(axis=1) + 50 * d[["rot_x", "rot_y", "rot_z"]].sum(axis=1)

print("FD 중앙값: %.3f mm, 최댓값: %.3f mm" % (fd.median(), fd.max()))
print("절대 위치의 최대 이동(z): %.2f mm" % motion["trans_z"].abs().max())
bad = np.where(fd > 0.5)[0]           # 흔히 쓰는 출발점 기준(데이터에 따라 0.2–0.9 mm까지 다양)
print("FD > 0.5 mm 인 시점:", bad.tolist())

# 문제 시점마다 '그 시점만 1'인 회귀변수(spike regressor)를 만들어 GLM에 넣을 수 있다
spikes = np.zeros((n, bad.size))
spikes[bad, np.arange(bad.size)] = 1
print("spike 회귀변수 행렬:", spikes.shape, "| 남는 시점 비율: %.1f%%" % (100 * (1 - bad.size / n)))

그림으로 남기려면 아래를 이어 붙입니다(그림 파일만 저장하고 화면 출력은 없습니다).

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path

Path("figures/advanced/D").mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
ax[0].plot(motion[["trans_x", "trans_y", "trans_z"]]); ax[0].set_ylabel("translation (mm)")
ax[1].plot(fd, color="k"); ax[1].axhline(0.5, ls="--", color="r"); ax[1].set_ylabel("FD (mm)")
ax[1].set_xlabel("volume (TR)")
fig.tight_layout(); fig.savefig("figures/advanced/D/D2_framewise_displacement.png", dpi=150)

### 예시 3 — smoothing 크기에 따라 넓은 활성과 작은 활성이 어떻게 달라지나

In [ ]:
import numpy as np
from scipy import ndimage

def fwhm_to_sigma(fwhm):
    return fwhm / (2 * np.sqrt(2 * np.log(2)))    # FWHM ≈ 2.355 σ

voxel = 3.0                                          # mm
rng = np.random.default_rng(3)
yy, xx = np.mgrid[0:64, 0:64]
big = 1.0 * ((xx - 20) ** 2 + (yy - 32) ** 2 < 8 ** 2)    # 반지름 8 voxel(24 mm)짜리 넓은 활성
small = 1.0 * ((xx - 48) ** 2 + (yy - 32) ** 2 < 1.5 ** 2)  # 반지름 1.5 voxel짜리 작은 활성
truth = big + small
noisy = truth + rng.normal(0, 1.0, truth.shape)

print("FWHM(mm)  넓은 활성 SNR  작은 활성 SNR")
for fwhm_mm in [0, 6, 12, 24, 36]:
    img = noisy if fwhm_mm == 0 else ndimage.gaussian_filter(noisy, fwhm_to_sigma(fwhm_mm / voxel))
    noise_sd = img[truth == 0].std()
    snr_big = img[big == 1].mean() / noise_sd
    snr_small = img[small == 1].mean() / noise_sd
    print(f"{fwhm_mm:>7}   {snr_big:12.2f}   {snr_small:12.2f}")

# 원래 있던 매끄러움과 일부러 준 smoothing은 제곱합의 제곱근으로 합쳐진다
intrinsic, applied = 4.0, 6.0
print("최종 매끄러움 ≈ %.1f mm (6 mm만 준 게 아니다)" % np.sqrt(intrinsic**2 + applied**2))

### 참고 — 실제 자료를 전처리할 때의 명령 형태 (실행 확인 안 됨, 패키지 미설치)

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# # Docker/Apptainer 컨테이너로 배포되는 fMRIPrep의 기본 호출 형태. 이 저장소 환경에서는 실행하지 않았다.
# # 입력은 BIDS 폴더, 출력 폴더, 처리 수준(participant)이다. FreeSurfer 라이선스 파일이 필요하다.
# fmriprep /path/to/bids /path/to/derivatives participant --participant-label 01 --fs-license-file /path/to/license.txt
# # 결과: derivatives/sub-01/func/ 아래의 전처리된 BOLD, *_desc-confounds_timeseries.tsv(FD 등), HTML 시각 보고서